In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import numpy as np

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Training History

Extract loss values from the training log to visualize convergence across epochs.

In [ ]:
# Parse training log
log_path = Path('tracking_baseline/train/training.log')
epochs = []
avg_losses = []

with open(log_path, 'r') as f:
    for line in f:
        if 'done in' in line and 'avg loss' in line:
            # Extract: "Epoch N done in Xs; avg loss Y"
            parts = line.split()
            epoch_idx = parts.index('Epoch') + 1
            loss_idx = parts.index('loss') + 1
            
            epoch = int(parts[epoch_idx])
            avg_loss = float(parts[loss_idx])
            
            epochs.append(epoch)
            avg_losses.append(avg_loss)

training_df = pd.DataFrame({
    'Epoch': epochs,
    'Average Loss': avg_losses
})

print("Training Summary:")
print(training_df.to_string(index=False))
print(f"\nFinal Loss: {avg_losses[-1]:.4f}")
print(f"Loss Reduction: {((avg_losses[0] - avg_losses[-1]) / avg_losses[0] * 100):.1f}%")

## 2. Visualize Training Convergence

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(training_df['Epoch'], training_df['Average Loss'], 
        marker='o', linewidth=2, markersize=8, color='#2E86AB')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Training Loss Convergence', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss decreased from {avg_losses[0]:.4f} to {avg_losses[-1]:.4f} over {len(epochs)} epochs")

## 3. Load Test Set Evaluation Metrics

Load the COCO metrics computed on the test set (36,750 images).

In [ ]:
# Load evaluation results
eval_path = Path('tracking_baseline/train/runs/frcnn/eval_metrics.json')

if eval_path.exists():
    with open(eval_path, 'r') as f:
        metrics = json.load(f)
    
    print("Test Set Performance (COCO Metrics):")
    print("=" * 50)
    for metric, value in metrics.items():
        print(f"{metric:15s}: {value:.4f} ({value*100:.2f}%)")
else:
    print(f"Evaluation metrics not found at {eval_path}")
    print("Run eval_coco.py first to generate metrics.")
    metrics = None

## 4. Visualize Detection Performance

In [ ]:
if metrics:
    # Create bar chart of metrics
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Overall AP metrics
    overall_metrics = ['AP', 'AP50', 'AP75']
    overall_values = [metrics[m] * 100 for m in overall_metrics]
    colors = ['#2E86AB', '#A23B72', '#F18F01']
    ye
    ax1.bar(overall_metrics, overall_values, color=colors, alpha=0.8)
    ax1.set_ylabel('Average Precision (%)', fontsize=11)
    ax1.set_title('Overall Detection Performance', fontsize=12, fontweight='bold')
    ax1.set_ylim([0, 100])
    for i, (metric, val) in enumerate(zip(overall_metrics, overall_values)):
        ax1.text(i, val + 2, f'{val:.1f}%', ha='center', fontsize=10)
    
    # Size-based AP metrics
    size_metrics = ['AP_small', 'AP_medium', 'AP_large']
    size_labels = ['Small (Ball)', 'Medium', 'Large (Players)']
    size_values = [metrics[m] * 100 for m in size_metrics]
    colors2 = ['#C73E1D', '#6A994E', '#386FA4']
    
    ax2.bar(size_labels, size_values, color=colors2, alpha=0.8)
    ax2.set_ylabel('Average Precision (%)', fontsize=11)
    ax2.set_title('Performance by Object Size', fontsize=12, fontweight='bold')
    ax2.set_ylim([0, 100])
    ax2.tick_params(axis='x', rotation=15)
    for i, (label, val) in enumerate(zip(size_labels, size_values)):
        ax2.text(i, val + 2, f'{val:.1f}%', ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Analysis
    print("\nKey Insights:")
    print("=" * 50)
    print(f"• Overall AP: {metrics['AP']*100:.2f}% - Average across all IoU thresholds (50%-95%)")
    print(f"• AP50: {metrics['AP50']*100:.2f}% - Easier threshold (50% overlap required)")
    print(f"• AP75: {metrics['AP75']*100:.2f}% - Stricter threshold (75% overlap required)")
    print(f"\n• Ball Detection (AP_small): {metrics['AP_small']*100:.2f}%")
    print(f"• Player Detection (AP_large): {metrics['AP_large']*100:.2f}%")
    
    if metrics['AP_small'] < metrics['AP_large']:
        print("\n⚠️ Ball detection is harder than player detection (expected - smaller objects)")
    if metrics['AP'] > 0.5:
        print("✅ Model shows good overall detection performance (AP > 50%)")
    if metrics['AP_small'] > 0.3:
        print("✅ Ball detection heuristic labeling was effective (AP_small > 30%)")

## 5. Compare Checkpoints Across Epochs

Optionally, you can evaluate earlier epoch checkpoints to see how performance improved during training.

In [ ]:
# Check available checkpoints
checkpoint_dir = Path('tracking_baseline/train/runs/frcnn')
checkpoints = sorted(checkpoint_dir.glob('model_epoch*.pth'))
checkpoint_files = [c.name for c in checkpoints if '_step' not in c.name]

print(f"Available epoch checkpoints: {len(checkpoint_files)}")
for ckpt in checkpoint_files:
    ckpt_path = checkpoint_dir / ckpt
    size_mb = ckpt_path.stat().st_size / (1024 * 1024)
    print(f"  • {ckpt:20s} ({size_mb:.1f} MB)")

print("\nNote: To evaluate earlier epochs, run eval_coco.py with --checkpoint pointing to that epoch.")
print("This shows how the model improved during training.")

## 6. Model Architecture Summary

In [ ]:
# Load final checkpoint to inspect model
final_checkpoint = checkpoint_dir / 'model_epoch6.pth'

if final_checkpoint.exists():
    ckpt = torch.load(final_checkpoint, map_location='cpu')
    
    print("Model Configuration:")
    print("=" * 50)
    print(f"Architecture: Faster R-CNN with ResNet-50 FPN backbone")
    print(f"Classes: 2 (Player=0, Ball=1)")
    print(f"Final Epoch: {ckpt.get('epoch', 'N/A')}")
    print(f"\nCheckpoint Contents:")
    for key in ckpt.keys():
        if key == 'model':
            num_params = sum(p.numel() for p in ckpt['model'].values() if isinstance(p, torch.Tensor))
            print(f"  • {key}: {num_params:,} parameters")
        else:
            print(f"  • {key}")
else:
    print(f"Checkpoint not found: {final_checkpoint}")

## 7. Training Configuration

In [ ]:
config_path = checkpoint_dir / 'config.json'

if config_path.exists():
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    print("Training Hyperparameters:")
    print("=" * 50)
    for key, value in config.items():
        if key not in ['data_root', 'coco_json']:
            print(f"{key:20s}: {value}")
    
    print(f"\nDataset Size:")
    print(f"  Training images: ~42,000")
    print(f"  Test images: ~36,750")
else:
    print(f"Config not found: {config_path}")

## 8. Visual Comparison: Predictions vs Ground Truth

Visualize model predictions on a test sequence with ground truth boxes for comparison.

In [ ]:
# ============ USER CONFIGURATION ============
# Change this to any test sequence you want to visualize
SEQUENCE_NAME = "SNMOT-116"  # Test sequences: SNMOT-116 to SNMOT-150, SNMOT-187 to SNMOT-200
# ===========================================

import torch
import torchvision
from torchvision import transforms as T
from PIL import Image
import cv2
import time
from collections import defaultdict

# Confidence threshold for displaying predictions
CONF_THRESHOLD = 0.5

# Load the trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = checkpoint_dir / 'model_epoch6.pth'

print(f"Loading model from {model_path.name}...")
checkpoint = torch.load(model_path, map_location=device)

# Rebuild model architecture
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()

print(f"Model loaded on {device}")

In [ ]:
# Find the test sequence
def find_folder_downwards(target_folder: str, start_dir: Path) -> Path | None:
    for path in start_dir.rglob(target_folder):
        if path.is_dir():
            return path
    return None

data_root = Path('data/tracking-2023/test')
seq_path = find_folder_downwards(SEQUENCE_NAME, data_root)

if seq_path is None:
    raise FileNotFoundError(f"Sequence '{SEQUENCE_NAME}' not found in {data_root}")

print(f"Found sequence: {seq_path}")

# Load ground truth annotations
imgdir = seq_path / "img1"
gt_txt = seq_path / "gt" / "gt.txt"

per_frame_gt = defaultdict(list)
if gt_txt.is_file():
    with open(gt_txt) as f:
        for line in f:
            if not line.strip():
                continue
            fr, tid, x, y, w, h, conf, *rest = line.strip().split(",")
            cls_ = int(float(rest[0])) if rest else 1  # class: 0=player, 1=ball
            per_frame_gt[int(float(fr))].append({
                'tid': int(float(tid)),
                'class': cls_,
                'bbox': [int(float(x)), int(float(y)), int(float(w)), int(float(h))]
            })

print(f"Loaded ground truth for {len(per_frame_gt)} frames")

In [ ]:
# Playback with predictions vs ground truth
win = "Model Evaluation - Predictions vs Ground Truth"
target_fps = 25
frame_period = 1.0 / target_fps

# Transform for model input
transform = T.Compose([T.ToTensor()])

# OpenCV video reader
cap = cv2.VideoCapture(str(imgdir / "%06d.jpg"))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open image sequence at {imgdir}")

window_created = False
frame_idx = 1

print("Starting playback...")
print("- GREEN boxes = Ground Truth (Player=solid, Ball=dashed)")
print("- RED boxes = Model Predictions (Player=solid, Ball=dashed)")
print("- Press 'q' or ESC to stop")
print()

try:
    while True:
        t0 = time.perf_counter()
        
        ok, img = cap.read()
        if not ok or img is None:
            break
        
        # Run model prediction
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = transform(Image.fromarray(img_rgb)).unsqueeze(0).to(device)
        
        with torch.no_grad():
            predictions = model(img_tensor)[0]
        
        # Draw ground truth (GREEN)
        for obj in per_frame_gt.get(frame_idx, []):
            x, y, w, h = obj['bbox']
            cls = obj['class']
            tid = obj['tid']
            
            # Players = solid green, Balls = dashed green
            color = (0, 255, 0)  # Green
            if cls == 1:  # Ball
                # Dashed rectangle for balls
                cv2.rectangle(img, (x, y), (x + w, y + h), color, 2, lineType=cv2.LINE_4)
                cv2.putText(img, f"GT Ball {tid}", (x, max(0, y - 4)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
            else:  # Player
                cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv2.putText(img, f"GT {tid}", (x, max(0, y - 4)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
        
        # Draw predictions (RED)
        boxes = predictions['boxes'].cpu().numpy()
        scores = predictions['scores'].cpu().numpy()
        labels = predictions['labels'].cpu().numpy()
        
        for box, score, label in zip(boxes, scores, labels):
            if score < CONF_THRESHOLD:
                continue
            
            x1, y1, x2, y2 = box.astype(int)
            color = (0, 0, 255)  # Red
            
            if label == 1:  # Ball
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2, lineType=cv2.LINE_4)
                cv2.putText(img, f"Pred Ball {score:.2f}", (x1, max(0, y1 - 4)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
            else:  # Player
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, f"Pred {score:.2f}", (x1, max(0, y1 - 4)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
        
        # Add frame counter
        cv2.putText(img, f"Frame: {frame_idx}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Create window when first frame is ready
        if not window_created:
            cv2.namedWindow(win, cv2.WINDOW_AUTOSIZE)
            window_created = True
        
        cv2.imshow(win, img)
        
        # Adaptive delay to maintain target FPS
        elapsed = time.perf_counter() - t0
        remaining = frame_period - elapsed
        delay_ms = 1 if remaining <= 0 else int(remaining * 1000)
        
        key = cv2.waitKey(delay_ms) & 0xFF
        if key in (27, ord('q')):  # ESC or q
            break
        
        frame_idx += 1

finally:
    cap.release()
    cv2.destroyWindow(win)
    cv2.waitKey(1)
    print(f"\nPlayback complete. Processed {frame_idx - 1} frames.")

**Color Coding:**
- **GREEN** = Ground Truth annotations (from `gt.txt`)
- **RED** = Model Predictions (confidence ≥ 0.5)
- **Solid lines** = Players (class 0)
- **Dashed/labeled** = Balls (class 1)

You can change `SEQUENCE_NAME` in the configuration cell above to visualize any test sequence (SNMOT-116 through SNMOT-150, or SNMOT-187 through SNMOT-200).

## Summary

This notebook provides comprehensive evaluation of the trained Faster R-CNN model for football player and ball detection:

**Training Analysis:**
- Visualizes loss convergence across 6 epochs showing model learning progression
- Tracks improvement from initial to final training state

**Performance Metrics:**
- **COCO AP metrics** measure detection accuracy on unseen test data (36,750 images)
- **AP50/AP75** show performance at different intersection-over-union thresholds
- **AP_small** specifically evaluates ball detection (challenging due to small object size)
- **AP_large** measures player detection performance

**Visual Validation:**
- Side-by-side comparison of model predictions vs ground truth on actual video sequences
- Color-coded bounding boxes for easy interpretation
- Configurable to analyze any test sequence

**Ball Labeling Approach:**
The model was trained using heuristic-based ball identification:
- Size constraint: <2000 px² and below 30th percentile
- Shape constraint: aspect ratio > 0.7 (near-circular)
- Temporal consistency: persistent across multiple frames

This evaluation framework allows complete assessment of model performance on the SoccerNet tracking dataset.